# 1: Install libraries


In [ ]:
!pip install -q transformers datasets evaluate accelerate sentencepiece sacrebleu rouge_score bert_score nltk pandas scikit-learn openpyxl

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.7 MB/s eta 0:00:00


# 2: Import libraries

In [ ]:
import os
import re
import gc
import torch
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

import evaluate
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

nltk.download("wordnet")
nltk.download("omw-1.4")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


GPU available: True
GPU: Tesla T4


# 3: Upload dataset

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_path = list(uploaded.keys())[0]

print("Uploaded file:", csv_path)

Saving Counterspeech_dataset_Old_cleaned (1).csv to Counterspeech_dataset_Old_cleaned (1).csv
Uploaded file: Counterspeech_dataset_Old_cleaned (1).csv


# 4: Load dataset

In [ ]:
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Dataset shape: (3011, 5)
Columns: ['id', 'offensive_text', 'counterspeech_text', 'counterspeech_type', 'text_type']


,id,offensive_text,counterspeech_text,counterspeech_type,text_type
0,1,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...,fact_based,hate_speech
1,2,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...,warning_of_consequences,sexual_harassment
2,3,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।,empathy,hate_speech
3,4,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...,fact_based,gender_abuse
4,5,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।,moral_appeal,abusive_language


# 5: Keep only generation columns

In [ ]:
required_columns = ["offensive_text", "counterspeech_text"]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

df = df[required_columns].copy()

df.head()

,offensive_text,counterspeech_text
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...
1,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...
2,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।
3,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...
4,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।


# 6: Clean text

In [ ]:
def clean_text(text):
    text = str(text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("\u200c", "")
    text = text.replace("\u200d", "")
    return text

df["offensive_text"] = df["offensive_text"].apply(clean_text)
df["counterspeech_text"] = df["counterspeech_text"].apply(clean_text)

df = df.dropna()
df = df.drop_duplicates()
df = df[df["offensive_text"].str.len() > 2]
df = df[df["counterspeech_text"].str.len() > 2]

print("Cleaned dataset shape:", df.shape)
df.head()

Cleaned dataset shape: (3011, 2)


,offensive_text,counterspeech_text
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...
1,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...
2,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।
3,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...
4,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।


# 7: Create GPT-style prompt and target

In [ ]:
def create_prompt(offensive_text):
    return (
        "নিচের আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপদ, "
        "অ-আক্রমণাত্মক এবং গালিবিহীন বাংলা প্রতিউত্তর লিখুন।\n\n"
        f"আপত্তিকর বক্তব্য: {offensive_text}\n"
        "ভদ্র প্রতিউত্তর:"
    )

df["prompt_text"] = df["offensive_text"].apply(create_prompt)
df["target_text"] = df["counterspeech_text"]

df = df[["prompt_text", "target_text"]]

df.head()

,prompt_text,target_text
0,"নিচের আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র,...",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...
1,"নিচের আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র,...",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...
2,"নিচের আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র,...",পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।
3,"নিচের আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র,...",এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...
4,"নিচের আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র,...",ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।


# 8: Train-validation-test split

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    shuffle=True
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

print("Train size:", len(train_df))
print("Validation size:", len(valid_df))
print("Test size:", len(test_df))

Train size: 2408
Validation size: 301
Test size: 302


# 9: Convert to Hugging Face Dataset

In [ ]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(valid_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True))
})

dataset

DatasetDict({
    train: Dataset({
        features: ['prompt_text', 'target_text'],
        num_rows: 2408
    })
    validation: Dataset({
        features: ['prompt_text', 'target_text'],
        num_rows: 301
    })
    test: Dataset({
        features: ['prompt_text', 'target_text'],
        num_rows: 302
    })
})

# Load BanglaGPT / Bangla-GPT2

In [ ]:
MODEL_NAME = "flax-community/gpt2-bengali"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

model.to(device)

print("Model:", MODEL_NAME)
print("Vocab size:", len(tokenizer))
print("Pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("EOS token:", tokenizer.eos_token, tokenizer.eos_token_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/864 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.76M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: flax-community/gpt2-bengali
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: flax-community/gpt2-bengali
Vocab size: 50256
Pad token: <|endoftext|> 0
EOS token: <|endoftext|> 0


# 11: Tokenize dataset with prompt masking

In [ ]:
MAX_PROMPT_LENGTH = 128
MAX_TARGET_LENGTH = 64
MAX_LENGTH = MAX_PROMPT_LENGTH + MAX_TARGET_LENGTH

def preprocess_function(examples):
    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for prompt, target in zip(examples["prompt_text"], examples["target_text"]):
        prompt = str(prompt).strip()
        target = str(target).strip()

        # Tokenize prompt separately
        prompt_tokens = tokenizer(
            prompt,
            max_length=MAX_PROMPT_LENGTH,
            truncation=True,
            padding=False
        )

        # Tokenize target separately
        target_text = " " + target + tokenizer.eos_token

        target_tokens = tokenizer(
            target_text,
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
            padding=False
        )

        input_ids = prompt_tokens["input_ids"] + target_tokens["input_ids"]
        attention_mask = [1] * len(input_ids)

        labels = [-100] * len(prompt_tokens["input_ids"]) + target_tokens["input_ids"]

        # Pad manually
        pad_len = MAX_LENGTH - len(input_ids)

        if pad_len > 0:
            input_ids = input_ids + [tokenizer.pad_token_id] * pad_len
            attention_mask = attention_mask + [0] * pad_len
            labels = labels + [-100] * pad_len
        else:
            input_ids = input_ids[:MAX_LENGTH]
            attention_mask = attention_mask[:MAX_LENGTH]
            labels = labels[:MAX_LENGTH]

        # Safety check: if labels became empty, skip by using EOS as label
        valid_label_count = sum(1 for x in labels if x != -100)

        if valid_label_count == 0:
            labels[-1] = tokenizer.eos_token_id
            attention_mask[-1] = 1

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list
    }


tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["prompt_text", "target_text"]
)

tokenized_dataset

Map:   0%|          | 0/2408 [00:00<?, ? examples/s]

Map:   0%|          | 0/301 [00:00<?, ? examples/s]

Map:   0%|          | 0/302 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2408
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 301
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 302
    })
})

# 12: Debug tokenization


In [ ]:
sample = tokenized_dataset["train"][0]

decoded_input = tokenizer.decode(
    [x for x in sample["input_ids"] if x != tokenizer.pad_token_id],
    skip_special_tokens=True
)

valid_label_ids = [x for x in sample["labels"] if x != -100]

decoded_label = tokenizer.decode(
    valid_label_ids,
    skip_special_tokens=True
)

print("Decoded full input:")
print(decoded_input)

print("\nDecoded label part only:")
print(decoded_label)

print("\nValid label token count:", len(valid_label_ids))

Decoded full input:
নিচের আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র, নিরাপদ, অ-আক্রমণাত্মক এবং গালিবিহীন বাংলা প্রতিউত্তর লিখুন।

আপত্তিকর বক্তব্য: নারীরা এগিয়ে যাচ্ছে, কিন্তু কোথায় সেটা বললে হুরের ভিন্ন মতের মানুষকে হেয় না করে আলোচনা করুন। অসম্মানজনক ভাষা ছাড়া বিতর্ক অনেক বেশি ফলপ্রসূ হ�

Decoded label part only:
 ভিন্ন মতের মানুষকে হেয় না করে আলোচনা করুন। অসম্মানজনক ভাষা ছাড়া বিতর্ক অনেক বেশি ফলপ্রসূ হ�

Valid label token count: 64


# 13: Batch loss test

In [ ]:
from torch.utils.data import DataLoader

small_train_dataset = tokenized_dataset["train"].select(range(4))

def causal_lm_collator(features):
    batch = {}

    for key in features[0].keys():
        batch[key] = torch.tensor([f[key] for f in features], dtype=torch.long)

    return batch

test_loader = DataLoader(
    small_train_dataset,
    batch_size=2,
    collate_fn=causal_lm_collator
)

batch = next(iter(test_loader))
batch = {k: v.to(device) for k, v in batch.items()}

print("Batch input_ids shape:", batch["input_ids"].shape)
print("Batch labels shape:", batch["labels"].shape)

non_ignored_labels = (batch["labels"] != -100).sum().item()
print("Non-ignored label tokens:", non_ignored_labels)

with torch.no_grad():
    outputs = model(**batch)

print("Test batch loss:", outputs.loss.item())

Batch input_ids shape: torch.Size([2, 192])
Batch labels shape: torch.Size([2, 192])
Non-ignored label tokens: 113


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Test batch loss: 2.092041015625


# 14: Training arguments

In [ ]:
OUTPUT_DIR = "/content/banglagpt_counterspeech_model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="no",
    save_strategy="epoch",

    learning_rate=3e-5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=8,

    num_train_epochs=8,

    weight_decay=0.01,

    fp16=False,

    logging_steps=25,

    save_total_limit=2,

    report_to="none",

    seed=SEED
)

# 15: Create Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],

    processing_class=tokenizer,
    data_collator=causal_lm_collator
)

# 16: Train BanglaGPT

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 0, 'bos_token_id': 0, 'pad_token_id': 0}.


Step,Training Loss
25,2.369643
50,2.075235
75,1.900070
100,1.786347
125,1.654057
150,1.585209
175,1.515546
200,1.431832
225,1.372271
250,1.342175


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1208, training_loss=1.1693021884027697, metrics={'train_runtime': 1609.686, 'train_samples_per_second': 11.968, 'train_steps_per_second': 0.75, 'total_flos': 1887573639168000.0, 'train_loss': 1.1693021884027697, 'epoch': 8.0})

# 17: Evaluate loss


In [ ]:
trainer.evaluate()

ValueError: Trainer: evaluation requires an eval_dataset.

# 18: Save model


In [ ]:
FINAL_MODEL_DIR = "/content/final_banglagpt_counterspeech"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Model saved to:", FINAL_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/final_banglagpt_counterspeech


# 19: Manual generation test

In [ ]:
model.eval()

def clean_generated_output(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)

    stop_markers = [
        "আপত্তিকর বক্তব্য:",
        "ভদ্র প্রতিউত্তর:",
        "Reference:",
        "Generated:",
        "Offensive:",
        "Prompt:"
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0].strip()

    # Remove repeated punctuation/spaces
    text = re.sub(r"[।]{2,}", "।", text)
    text = re.sub(r"\s+", " ", text).strip()

    # Keep output within reasonable length: first 18 words
    words = text.split()
    if len(words) > 18:
        text = " ".join(words[:18])

    return text


def generate_counterspeech(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=MAX_PROMPT_LENGTH,
        truncation=True
    ).to(device)

    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,

            max_new_tokens=50,
            min_new_tokens=6,

            do_sample=True,
            temperature=0.65,
            top_p=0.85,

            repetition_penalty=1.30,
            no_repeat_ngram_size=3,

            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][input_len:]

    generated_part = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    generated_part = clean_generated_output(generated_part)

    return generated_part


for i in range(20):
    prompt = test_df["prompt_text"].iloc[i]
    ref = test_df["target_text"].iloc[i]
    pred = generate_counterspeech(prompt)

    print("Reference:", ref)
    print("Generated:", pred)
    print("Generated length:", len(pred.split()))
    print("-" * 80)

Reference: এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথে কাজ করছে।
Generated: দেশ ও ধর্ম নিড়ের সম্পর্ক টেনে এমন কথা বলা খুবই অপমানজনক; যা রাজনৈতিক মতাদর্শকে
Generated length: 14
--------------------------------------------------------------------------------
Reference: প্রতারক বললে সমস্যার সমাধান হয় না; নির্দিষ্ট আচরণ উল্লেখ করলে কথা শক্ত হয়।
Generated: াওয়; এমন কথা বললে, সমস্যা আরও বড় হড়ে।
Generated length: 8
--------------------------------------------------------------------------------
Reference: মানুষকে নষ্ট বলে দাগিয়ে দিলে বোঝাপড়া কমে যায়, ভদ্র ভাষাই ভালো।
Generated: কাউকে আঘাত করা ঠিক নয়। সম্মানজনক ভাষা দেখানয়; কথা বলা দরকার।
Generated length: 11
--------------------------------------------------------------------------------
Reference: একটি জাতিকে শুয়োরের সঙ্গে তুলনা করা ঘৃণা ছড়ায়।
Generated: কারও পরিচয় নিড়ি७ে, তাকে অপমান করা ঠিক নয়।
Generated length: 8
--------------------------------------------------------------------------------
Reference: বিশ্বাসভিন্ন মানুষদের নিয়ে 

# 20: Generate full test predictions

In [ ]:
test_inputs = test_df["prompt_text"].tolist()
references = test_df["target_text"].tolist()

predictions = []

for i, prompt in enumerate(test_inputs):
    pred = generate_counterspeech(prompt)

    # Fallback if output is empty
    if len(pred.strip()) == 0:
        pred = "সম্মানজনক ভাষায় যুক্তি দিয়ে কথা বলা উচিত।"

    predictions.append(pred)

    if i % 50 == 0:
        print(f"Generated {i}/{len(test_inputs)}")

results_df = pd.DataFrame({
    "input_text": test_inputs,
    "reference_counterspeech": references,
    "generated_counterspeech": predictions
})

results_df["generated_length"] = results_df["generated_counterspeech"].apply(
    lambda x: len(str(x).split())
)

results_df.head(20)

Generated 0/302
Generated 50/302
Generated 100/302
Generated 150/302
Generated 200/302
Generated 250/302
Generated 300/302


,input_text,reference_counterspeech,generated_counterspeech,generated_length
0,"নিচের আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র,...",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...,"অপমান করে কথা বললে, অপমানটাই শুধু কমে।",7
1,"নিচের আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র,...",প্রতারক বললে সমস্যার সমাধান হয় না; নির্দিষ্ট আ...,"াওয় কথা বললে, সমাধান আসে না।",6
2,"নিচের আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র,...","মানুষকে নষ্ট বলে দাগিয়ে দিলে বোঝাপড়া কমে যায়, ...",কাউকে অপমান করলে পরিস্থিতি আরও খারাপ হয়।,7
3,"নিচের আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র,...",একটি জাতিকে শুয়োরের সঙ্গে তুলনা করা ঘৃণা ছড়ায়।,কাউকে দোষারোপ করলে তাকে ছোট করা হয়।,7
4,"নিচের আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র,...",বিশ্বাসভিন্ন মানুষদের নিয়ে এভাবে বিদ্রূপ করা ...,"োষ্ঠীবদ্ধ আচরণ করতে চাইলে, সম্পর্কটা নষ্ট হয়, ...",11
5,"নিচের আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র,...",এভাবে আঘাতের কথা বলা আরও ক্ষতি বাড়াবে।,তাও আক্রমণ করা যা৛; সমস্যা থাকলে সেটা শান্তভাব...,10
6,"নিচের আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র,...",কারও পরিবার নিয়ে এমন কটাক্ষ খুবই অশালীন এবং কষ...,কারও পরিবারকে টেনে এমন অপমানজনক কথা বলা খুবই ক...,14
7,"নিচের আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র,...",কারও মতকে সমর্থন করলেই তাকে বুদ্ধি প্রতিবন্ধী ...,"দৌড়ি করে, সেই সব কথা বললে সমস্যা আরও জটিলতর হয়।",10
8,"নিচের আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র,...","কারও মৃত্যু কামনা করা মানসিক ক্ষতি ডেকে আনে, ত...",কারও মৃত্যুকে কেন্দ্র করে অশালীন ভাষা ব্যবহার ...,13
9,"নিচের আপত্তিকর বাংলা বক্তব্যের জন্য একটি ভদ্র,...",জাতিগত অপমান বাদ দিলে তর্কটা অনেক বেশি পরিষ্কা...,"কারও শরীর নিয়ে এমন হুমকি, এটা খুবই অশোভন ও অমা...",10


# 21: Check generated length

In [ ]:
print(results_df["generated_length"].describe())

results_df[["reference_counterspeech", "generated_counterspeech", "generated_length"]].head(20)

count    302.000000
mean      10.304636
std        3.082462
min        2.000000
25%        8.000000
50%       11.000000
75%       13.000000
max       18.000000
Name: generated_length, dtype: float64


,reference_counterspeech,generated_counterspeech,generated_length
0,এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...,"অপমান করে কথা বললে, অপমানটাই শুধু কমে।",7
1,প্রতারক বললে সমস্যার সমাধান হয় না; নির্দিষ্ট আ...,"াওয় কথা বললে, সমাধান আসে না।",6
2,"মানুষকে নষ্ট বলে দাগিয়ে দিলে বোঝাপড়া কমে যায়, ...",কাউকে অপমান করলে পরিস্থিতি আরও খারাপ হয়।,7
3,একটি জাতিকে শুয়োরের সঙ্গে তুলনা করা ঘৃণা ছড়ায়।,কাউকে দোষারোপ করলে তাকে ছোট করা হয়।,7
4,বিশ্বাসভিন্ন মানুষদের নিয়ে এভাবে বিদ্রূপ করা ...,"োষ্ঠীবদ্ধ আচরণ করতে চাইলে, সম্পর্কটা নষ্ট হয়, ...",11
5,এভাবে আঘাতের কথা বলা আরও ক্ষতি বাড়াবে।,তাও আক্রমণ করা যা৛; সমস্যা থাকলে সেটা শান্তভাব...,10
6,কারও পরিবার নিয়ে এমন কটাক্ষ খুবই অশালীন এবং কষ...,কারও পরিবারকে টেনে এমন অপমানজনক কথা বলা খুবই ক...,14
7,কারও মতকে সমর্থন করলেই তাকে বুদ্ধি প্রতিবন্ধী ...,"দৌড়ি করে, সেই সব কথা বললে সমস্যা আরও জটিলতর হয়।",10
8,"কারও মৃত্যু কামনা করা মানসিক ক্ষতি ডেকে আনে, ত...",কারও মৃত্যুকে কেন্দ্র করে অশালীন ভাষা ব্যবহার ...,13
9,জাতিগত অপমান বাদ দিলে তর্কটা অনেক বেশি পরিষ্কা...,"কারও শরীর নিয়ে এমন হুমকি, এটা খুবই অশোভন ও অমা...",10


# 22: Save predictions

In [ ]:
prediction_csv = "/content/banglagpt_test_predictions.csv"

results_df.to_csv(prediction_csv, index=False, encoding="utf-8-sig")

print("Saved predictions:", prediction_csv)

files.download(prediction_csv)

Saved predictions: /content/banglagpt_test_predictions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 23: Metric helper functions

In [ ]:
smooth = SmoothingFunction().method1

def tokenize_bn(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.split()


def calculate_bleu_scores(predictions, references):
    bleu_1_scores = []
    bleu_2_scores = []
    bleu_3_scores = []
    bleu_4_scores = []

    for pred, ref in zip(predictions, references):
        pred_tokens = tokenize_bn(pred)
        ref_tokens = tokenize_bn(ref)

        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            bleu_1_scores.append(0)
            bleu_2_scores.append(0)
            bleu_3_scores.append(0)
            bleu_4_scores.append(0)
            continue

        bleu_1_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(1, 0, 0, 0),
            smoothing_function=smooth
        ))

        bleu_2_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(0.5, 0.5, 0, 0),
            smoothing_function=smooth
        ))

        bleu_3_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(1/3, 1/3, 1/3, 0),
            smoothing_function=smooth
        ))

        bleu_4_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smooth
        ))

    return {
        "BLEU-1": float(np.mean(bleu_1_scores)),
        "BLEU-2": float(np.mean(bleu_2_scores)),
        "BLEU-3": float(np.mean(bleu_3_scores)),
        "BLEU-4": float(np.mean(bleu_4_scores))
    }


def calculate_meteor(predictions, references):
    scores = []

    for pred, ref in zip(predictions, references):
        pred_tokens = tokenize_bn(pred)
        ref_tokens = tokenize_bn(ref)

        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            scores.append(0)
            continue

        try:
            score = meteor_score([ref_tokens], pred_tokens)
        except:
            score = 0

        scores.append(score)

    return float(np.mean(scores))


def get_ngrams(tokens, n):
    return list(zip(*[tokens[i:] for i in range(n)]))


def calculate_diversity(predictions):
    all_unigrams = []
    all_bigrams = []

    for text in predictions:
        tokens = tokenize_bn(text)
        all_unigrams.extend(tokens)
        all_bigrams.extend(get_ngrams(tokens, 2))

    distinct_1 = len(set(all_unigrams)) / len(all_unigrams) if len(all_unigrams) > 0 else 0
    distinct_2 = len(set(all_bigrams)) / len(all_bigrams) if len(all_bigrams) > 0 else 0

    diversity = (distinct_1 + distinct_2) / 2

    return {
        "Distinct-1": distinct_1,
        "Distinct-2": distinct_2,
        "Diversity": diversity
    }


def normalize_bn_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def token_jaccard_similarity(a, b):
    a_tokens = set(tokenize_bn(normalize_bn_text(a)))
    b_tokens = set(tokenize_bn(normalize_bn_text(b)))

    if len(a_tokens) == 0 or len(b_tokens) == 0:
        return 0.0

    intersection = len(a_tokens.intersection(b_tokens))
    union = len(a_tokens.union(b_tokens))

    return intersection / union if union > 0 else 0.0


def calculate_novelty(predictions, train_targets, similarity_threshold=0.70):
    train_targets = [normalize_bn_text(x) for x in train_targets]
    predictions = [normalize_bn_text(x) for x in predictions]

    max_similarities = []
    near_copy_count = 0

    for pred in predictions:
        if len(pred.strip()) == 0:
            max_similarities.append(0.0)
            continue

        similarities = [
            token_jaccard_similarity(pred, train_text)
            for train_text in train_targets
        ]

        max_similarity = max(similarities)
        max_similarities.append(max_similarity)

        if max_similarity >= similarity_threshold:
            near_copy_count += 1

    average_max_similarity = float(np.mean(max_similarities)) if len(max_similarities) > 0 else 0.0
    near_copy_rate = near_copy_count / len(predictions) if len(predictions) > 0 else 0.0

    novelty = 1 - average_max_similarity

    return {
        "Novelty": novelty,
        "Average Max Train Similarity": average_max_similarity,
        "Near Copy Rate": near_copy_rate
    }


bangla_abuse_words = [
    "চুদ",
    "মাগী",
    "হারামি",
    "শালা",
    "কুত্তা",
    "বাল",
    "খানকি",
    "বেশ্যা",
    "মাদারচোদ",
    "বোকাচোদা"
]


def calculate_abuse_word_rate(predictions, abuse_words):
    count = 0

    for text in predictions:
        text = str(text)
        found = any(word in text for word in abuse_words)

        if found:
            count += 1

    abuse_rate = count / len(predictions) if len(predictions) > 0 else 0

    return {
        "Abuse-word rate": abuse_rate,
        "Abusive outputs": count
    }


def calculate_length_stats(predictions):
    lengths = [len(tokenize_bn(text)) for text in predictions]

    empty_count = sum(1 for text in predictions if len(str(text).strip()) == 0)
    empty_rate = empty_count / len(predictions) if len(predictions) > 0 else 0

    return {
        "Average Generated Length": float(np.mean(lengths)),
        "Empty Output Rate": empty_rate
    }

# 24: Bengali ROUGE

In [ ]:
def simple_tokenize_bn(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.split()


def get_ngrams_list(tokens, n):
    if len(tokens) < n:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]


def rouge_n_score(pred, ref, n=1):
    pred_tokens = simple_tokenize_bn(pred)
    ref_tokens = simple_tokenize_bn(ref)

    pred_ngrams = get_ngrams_list(pred_tokens, n)
    ref_ngrams = get_ngrams_list(ref_tokens, n)

    if len(pred_ngrams) == 0 or len(ref_ngrams) == 0:
        return 0.0

    pred_counts = {}
    ref_counts = {}

    for ng in pred_ngrams:
        pred_counts[ng] = pred_counts.get(ng, 0) + 1

    for ng in ref_ngrams:
        ref_counts[ng] = ref_counts.get(ng, 0) + 1

    overlap = 0

    for ng in ref_counts:
        overlap += min(ref_counts[ng], pred_counts.get(ng, 0))

    recall = overlap / len(ref_ngrams)
    precision = overlap / len(pred_ngrams)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def lcs_length(x, y):
    m, n = len(x), len(y)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m):
        for j in range(n):
            if x[i] == y[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    return dp[m][n]


def rouge_l_score(pred, ref):
    pred_tokens = simple_tokenize_bn(pred)
    ref_tokens = simple_tokenize_bn(ref)

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    lcs = lcs_length(pred_tokens, ref_tokens)

    recall = lcs / len(ref_tokens)
    precision = lcs / len(pred_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def calculate_custom_rouge(predictions, references):
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    for pred, ref in zip(predictions, references):
        rouge1_scores.append(rouge_n_score(pred, ref, n=1))
        rouge2_scores.append(rouge_n_score(pred, ref, n=2))
        rougeL_scores.append(rouge_l_score(pred, ref))

    return {
        "rouge1": float(np.mean(rouge1_scores)),
        "rouge2": float(np.mean(rouge2_scores)),
        "rougeL": float(np.mean(rougeL_scores))
    }


rouge_scores = calculate_custom_rouge(predictions, references)

rouge_scores

{'rouge1': 0.07231382268500362,
 'rouge2': 0.009530398693848616,
 'rougeL': 0.0687510938604501}

#  25: BERTScore

In [ ]:
# ============================================================
# BERTScore Calculation - Fixed Version for BanglaGPT
# Uses bert_score package directly instead of evaluate.load("bertscore")
# This avoids XLMRobertaTokenizer compatibility error.
# ============================================================

!pip install -q bert_score

from bert_score import score as bert_score
import numpy as np

# Convert everything to clean string
clean_predictions = [str(x).strip() for x in predictions]
clean_references = [str(x).strip() for x in references]

# BERTScore cannot properly handle fully empty predictions,
# so replace empty outputs with a placeholder Bengali token.
clean_predictions = [
    pred if len(pred) > 0 else "ফাঁকা"
    for pred in clean_predictions
]

clean_references = [
    ref if len(ref) > 0 else "ফাঁকা"
    for ref in clean_references
]

try:
    P, R, F1 = bert_score(
        cands=clean_predictions,
        refs=clean_references,
        model_type="xlm-roberta-base",
        lang="bn",
        rescale_with_baseline=False,
        verbose=True,
        batch_size=16,
        use_fast_tokenizer=True
    )

    bertscore_precision = float(P.mean().item())
    bertscore_recall = float(R.mean().item())
    bertscore_f1 = float(F1.mean().item())

except Exception as e:
    print("XLM-R BERTScore failed. Error:")
    print(e)
    print("\nTrying fallback model: bert-base-multilingual-cased")

    P, R, F1 = bert_score(
        cands=clean_predictions,
        refs=clean_references,
        model_type="bert-base-multilingual-cased",
        lang="bn",
        rescale_with_baseline=False,
        verbose=True,
        batch_size=16,
        use_fast_tokenizer=True
    )

    bertscore_precision = float(P.mean().item())
    bertscore_recall = float(R.mean().item())
    bertscore_f1 = float(F1.mean().item())

print("BERTScore Precision:", bertscore_precision)
print("BERTScore Recall:", bertscore_recall)
print("BERTScore F1:", bertscore_f1)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/38 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/19 [00:00<?, ?it/s]

done in 1.41 seconds, 213.68 sentences/sec
BERTScore Precision: 0.8550891876220703
BERTScore Recall: 0.8546704649925232
BERTScore F1: 0.8547760248184204


# 26: Calculate final metrics

In [ ]:
bleu_scores = calculate_bleu_scores(predictions, references)

meteor = calculate_meteor(predictions, references)

diversity_scores = calculate_diversity(predictions)

novelty_scores = calculate_novelty(
    predictions,
    train_df["target_text"].tolist(),
    similarity_threshold=0.70
)

abuse_scores = calculate_abuse_word_rate(
    predictions,
    bangla_abuse_words
)

length_scores = calculate_length_stats(predictions)

final_metrics = {
    "Model": "BanglaGPT",

    "BLEU-1": bleu_scores["BLEU-1"],
    "BLEU-2": bleu_scores["BLEU-2"],
    "BLEU-3": bleu_scores["BLEU-3"],
    "BLEU-4": bleu_scores["BLEU-4"],

    "ROUGE-1": rouge_scores["rouge1"],
    "ROUGE-2": rouge_scores["rouge2"],
    "ROUGE-L": rouge_scores["rougeL"],

    "BERTScore Precision": bertscore_precision,
    "BERTScore Recall": bertscore_recall,
    "BERTScore F1": bertscore_f1,

    "Diversity": diversity_scores["Diversity"],
    "Distinct-1": diversity_scores["Distinct-1"],
    "Distinct-2": diversity_scores["Distinct-2"],

    "Novelty": novelty_scores["Novelty"],
    "Average Max Train Similarity": novelty_scores["Average Max Train Similarity"],
    "Near Copy Rate": novelty_scores["Near Copy Rate"],

    "Abuse-word rate": abuse_scores["Abuse-word rate"],
    "Abusive outputs": abuse_scores["Abusive outputs"],

    "METEOR": meteor,

    "Average Generated Length": length_scores["Average Generated Length"],
    "Empty Output Rate": length_scores["Empty Output Rate"]
}

metrics_df = pd.DataFrame([final_metrics])

metrics_df

,Model,BLEU-1,BLEU-2,BLEU-3,BLEU-4,ROUGE-1,ROUGE-2,ROUGE-L,BERTScore Precision,BERTScore Recall,...,Distinct-1,Distinct-2,Novelty,Average Max Train Similarity,Near Copy Rate,Abuse-word rate,Abusive outputs,METEOR,Average Generated Length,Empty Output Rate
0,BanglaGPT,0.060437,0.023574,0.014877,0.011575,0.072314,0.00953,0.068751,0.855089,0.85467,...,0.274743,0.614235,0.711403,0.288597,0.0,0.003311,1,0.043391,10.304636,0.0


# 27: Save metrics

In [ ]:
metrics_csv = "/content/banglagpt_final_metrics.csv"

metrics_df.to_csv(metrics_csv, index=False, encoding="utf-8-sig")

print("Saved metrics:", metrics_csv)

files.download(metrics_csv)

Saved metrics: /content/banglagpt_final_metrics.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 28: Zip and download model

In [ ]:
!zip -r /content/final_banglagpt_counterspeech.zip /content/final_banglagpt_counterspeech

files.download("/content/final_banglagpt_counterspeech.zip")

  adding: content/final_banglagpt_counterspeech/ (stored 0%)
  adding: content/final_banglagpt_counterspeech/training_args.bin (deflated 53%)
  adding: content/final_banglagpt_counterspeech/generation_config.json (deflated 31%)
  adding: content/final_banglagpt_counterspeech/config.json (deflated 52%)
  adding: content/final_banglagpt_counterspeech/model.safetensors (deflated 7%)
  adding: content/final_banglagpt_counterspeech/tokenizer.json (deflated 82%)
  adding: content/final_banglagpt_counterspeech/tokenizer_config.json (deflated 50%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>